In [ ]:

from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import solve_ivp
from scipy.special import expit


@dataclass
class JRParams:
    # Synaptic gains (mV)
    A: float = 3.25
    B: float = 22.0

    # Inverse synaptic time constants (1/s)
    a: float = 100.0
    b: float = 50.0

    # Connectivity constants
    C1: float = 135.0
    C2: float = 108.0
    C3: float = 33.75
    C4: float = 33.75

    # Sigmoid parameters
    e0: float = 2.5  # Maximum sigmoid firing rate = 2*e0
    v0: float = 6.0  # mV
    r: float = 0.56  # 1/mV


def sigmoid(v, p: JRParams):
    """Convert membrane potential (mV) to firing rate (1/s)."""
    return 2.0 * p.e0 * expit(p.r * (v - p.v0))


def jr_rhs(t, y, p: JRParams, external_input):
    """
    State:
        y[0:3] = y0, y1, y2 (mV)
        y[3:6] = dy0, dy1, dy2 (mV/s)

    external_input(t) returns the external firing rate (1/s).
    """
    y0, y1, y2, dy0, dy1, dy2 = y

    # Second derivatives of the three PSPs
    ddy0 = (
        p.A * p.a * sigmoid(y1 - y2, p)
        - 2 * p.a * dy0
        - p.a**2 * y0
    )

    ddy1 = (
        p.A * p.a * (
            external_input(t)
            + p.C2 * sigmoid(p.C1 * y0, p)
        )
        - 2 * p.a * dy1
        - p.a**2 * y1
    )

    ddy2 = (
        p.B * p.b * p.C4 * sigmoid(p.C3 * y0, p)
        - 2 * p.b * dy2
        - p.b**2 * y2
    )

    return np.array([dy0, dy1, dy2, ddy0, ddy1, ddy2])


def simulate_jr(
    duration=10.0,
    fs=1000.0,
    external_rate=120.0,
    params=None,
):
    """Simulate a single Jansen-Rit cortical column."""
    p = params or JRParams()

    t_eval = np.arange(0.0, duration, 1.0 / fs)
    y_init = np.zeros(6)

    solution = solve_ivp(
        jr_rhs,
        t_span=(0.0, duration),
        y0=y_init,
        t_eval=t_eval,
        args=(p, lambda t: external_rate),
        method="RK45",
        rtol=1e-6,
        atol=1e-8,
        max_step=1.0 / fs,
    )

    if not solution.success:
        raise RuntimeError(solution.message)

    # Net excitatory minus inhibitory PSP
    eeg = solution.y[1] - solution.y[2]

    return solution.t, eeg, solution.y


In [ ]:
from scipy.signal import welch # TODO unite with utils_spectral_power

def alpha_spectral_power(
    times: np.ndarray,
    signal: np.ndarray,
    sfreq: float,
    alpha_band=(8.0, 13.0),
    window_sec=4.0,
    step_sec=1.0,
):
    window_samples = int(round(window_sec * sfreq))
    step_samples = int(round(step_sec * sfreq))

    if len(signal) < window_samples:
        raise ValueError("Signal is shorter than one analysis window.")

    alpha_mask = None
    power_times = []
    alpha_power = []

    for start in range(
        0, len(signal) - window_samples + 1, step_samples
    ):
        segment = signal[start:start + window_samples]

        freqs, psd = welch(
            segment,
            fs=sfreq,
            nperseg=window_samples,
        )

        if alpha_mask is None:
            alpha_mask = (
                (freqs >= alpha_band[0])
                & (freqs <= alpha_band[1])
            )
            if np.count_nonzero(alpha_mask) < 2:
                raise ValueError(
                    "Insufficient frequency bins in the alpha band."
                )

        # Integrated PSD: power in mV².
        power = np.trapz(
            psd[alpha_mask],
            x=freqs[alpha_mask],
        )

        alpha_power.append(power)

        # Center of the current window, in seconds.
        center_sample = start + window_samples / 2
        power_times.append(times[0] + center_sample / sfreq)

    return np.asarray(power_times), np.asarray(alpha_power)


In [ ]:
t, eeg, states = simulate_jr(
    duration=20.0,
    fs=1000.0,
    external_rate=120.0,
)

power_times, alpha_power = alpha_spectral_power(
    t,
    eeg,
    sfreq=1000.0,
    alpha_band=(8.0, 13.0),
    window_sec=4.0,
    step_sec=1.0,
)

fig, axes = plt.subplots(
    2, 1, figsize=(12, 7), sharex=True
)

axes[0].plot(t, eeg, linewidth=0.7)
axes[0].set_ylabel("JR output (mV)")
axes[0].set_title("Jansen-Rit output and alpha-band power")
axes[0].grid(True, alpha=0.3)

axes[1].plot(
    power_times, alpha_power, marker="o", markersize=3
)
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel(r"Alpha power (mV$^2$)")
axes[1].set_title("Alpha spectral power (8–13 Hz)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()